# Experiments 52 - 54
Impact of applying exGreen techniques to create a derived datase _(false color images)_.

- **Model:**
    1. `yolov8m` *(Medium)*
- **Dataset:** 3.5m | 90º ***(v4i)***
    1. 2 PCA + exGreen + BurnBlend (exp 1 & 3)
    1. 2 PCA + exGreen + BurnBlend + CLAHE (exp 2)
- **Crop Tiles:** 640px
- **Sizes:** small & mid
- **Experiments:**
    1. 2 PCA + exGreen w/Bl
    1. Same + CLAHE
    1. Same w/o pre-train

    - **Reference:** RGB Exp 50 - *Freezing Backbone (10 layers)*

## Init

In [24]:
import os
import shutil
import fnmatch
import pickle
import torch

In [25]:
!pip install ultralytics

### Disabling augmentation

In [26]:
# IF default augmentation is not desiered, use the following line
# !pip uninstall albumentations

    # Disable all type of augmentation
    augment=False,
    erasing = 0,
    hsv_h=0,
    hsv_s=0,
    hsv_v=0,
    degrees=0.0,
    translate=0,
    scale=0.5,
    shear=0.0,
    flipud=0.0,
    fliplr=0.0

## Helper Functions

In [27]:
# Change for different file formats
reference = {
  "small": {
    "suffix": ".S",
    "file": "209"
  },
  "mid": {
    "suffix": ".M",
    "file": "503"
  },
  "large": {
    "suffix": ".L",
    "file": "000"
  }
}

In [28]:
# Clone config files
def copy_config(src_folder, dest_folder):
    """
    Copies files from src_folder to dest_folder, excluding subfolders.

    Args:
        src_folder: The path to the source folder.
        dest_folder: The path to the destination folder.
    """

    try:
        # Ensure destination folder exists
        os.makedirs(dest_folder, exist_ok=True)

        for filename in os.listdir(src_folder):
            src_path = os.path.join(src_folder, filename)
            dest_path = os.path.join(dest_folder, filename)

            if os.path.isfile(src_path):
                shutil.copy2(src_path, dest_path) #copy metadata as well.
                #Use shutil.copy for not copying metadata.
                print(f"Copied: {filename}")
            #else: #optional
                #print(f"Skipped (not a file): {filename}") #optional. Uncomment if you want to see the skipped folders.

        print("✅ Copying complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")


In [29]:
# Copy filtered dataset images/labels
def copy_and_filter_folder(src_folder, dest_folder, pattern):
    """
    Copies a folder and files that match the given pattern.
    Alerts the user when a folder or file already exists but *does not* overwrite.
    Creates only what is needed.

    :param src_folder: Path to the source folder.
    :param dest_folder: Path to the destination folder.
    :param pattern: Filename pattern to keep (e.g., "*.txt").
    """
    try:
        # Ensure destination folder exists
        if not os.path.exists(dest_folder):
            print(f"✓ Creating destination folder '{dest_folder}'.\n")
            os.makedirs(dest_folder)
        else:
            print(f"✓ Destination folder '{dest_folder}' already exists.\n")

        # Walk through the source folder
        for root, _, files in os.walk(src_folder):
            relative_path = os.path.relpath(root, src_folder)
            new_root = os.path.join(dest_folder, relative_path)

            if not os.path.exists(new_root):
                print(f"Creating subdirectory '{new_root}'")
                os.makedirs(new_root)
            else:
                print(f"❕Subdirectory '{new_root}' already exists.")
                print("Make sure the data inside is relevant. Otherwise, just delete the folder and repeat the cloning process.")

            for file in files:
                if fnmatch.fnmatch(file, pattern + "*"):
                    src_file = os.path.join(root, file)
                    dest_file = os.path.join(new_root, file)

                    if not os.path.exists(dest_file):
                        shutil.copy2(src_file, dest_file)  # copy metadata as well
                    else:
                        print(f"❗️File '{dest_file}' already exists. Skipping.")

            print(f" ✓ Copying files complete.\n")
        print("✅ Copying dataset complete.")

    except Exception as e:
        print(f"❌ An error occurred: {e}")

In [30]:
def copy_directory(source, destination, overwrite=True):
    try:
        if overwrite and os.path.exists(destination):
            shutil.rmtree(destination)  # Remove existing directory
        shutil.copytree(source, destination)
        print(f"Directory '{source}' copied to '{destination}'")
        return True  # Indicate success
    except FileExistsError:
        print(f"Destination '{destination}' exists. Use overwrite=True to replace.")
        return False  # Indicate failure

In [31]:
# Load/Download prediction results (BB + confidence) as .pkl file

def save_results(results, filename):
    with open(filename, 'wb') as f:
        pickle.dump(results, f)

def load_results(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [32]:
def save_on_cloud(source: str, destination: str):
    """
    Saves a folder to a cloud storage location (e.g., Google Drive in Colab).

    Args:
        source (str): The path to the source folder.
        destination (str): The path to the destination folder (in cloud storage).
    """
    # 0. Input Validation (Assertions)
    assert isinstance(source, str), "Source must be a string."
    assert isinstance(destination, str), "Destination must be a string."

    try:
        # 1. Verify Source Folder
        if not os.path.exists(destination):
            os.makedirs(destination)

        # 2. Copy the Folder
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print("✅ Folder copied successfully:\n  ",source,"\n  -->",destination)

    except Exception as e:
        print(f"❌ An error occurred: {e}")

# Datasets builder

## Importing from Drive

In [33]:
!rm -rf /content/sample_data

In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# Check if the cloud path is ok and the dataset can be found
!ls /content/drive/MyDrive/YOLO

3.5m.v3i.yolov8.640px		       3.5m.v4i.yolov8_blended.640px
3.5m.v3i.yolov8.640px.aug.v1	       3.5m.v4i.yolov8_blended.640px_clahe
3.5m.v3i.yolov8.640px.aug.v1.soil_aug  best_e26.pt
3.5m.v3i.yolov8.640px_clahe	       Inference
3.5m.v3i.yolov8.640px.soil_aug	       models
3.5m.v4i.yolov8.640px		       optuna_yolov8_f1_study.db
3.5m.v4i.yolov8.640px_aug5m


In [62]:
drive_path = '/content/drive/MyDrive/YOLO'
drive_datasets_paths = os.listdir(drive_path)
drive_datasets = len(drive_datasets_paths)
if (drive_datasets) > 1:
    print("There are %d dataset options:" % drive_datasets)
else:
    print("Theres is only 1 dataset:")
drive_datasets_paths

There are 13 dataset options:


['3.5m.v3i.yolov8.640px',
 'Inference',
 'models',
 '3.5m.v3i.yolov8.640px.aug.v1',
 'best_e26.pt',
 'optuna_yolov8_f1_study.db',
 '3.5m.v3i.yolov8.640px.soil_aug',
 '3.5m.v3i.yolov8.640px.aug.v1.soil_aug',
 '3.5m.v3i.yolov8.640px_clahe',
 '3.5m.v4i.yolov8.640px',
 '3.5m.v4i.yolov8.640px_aug5m',
 '3.5m.v4i.yolov8_blended.640px',
 '3.5m.v4i.yolov8_blended.640px_clahe']

`3.5m.v4i.yolov8_blended.640px`

`3.5m.v4i.yolov8_blended.640px_clahe`

In [ ]:
choose_dataset = 12
index = choose_dataset - 1
model_name = os.listdir(drive_path)[index]
print("Chosen model:", model_name)

Chosen model: 3.5m.v4i.yolov8_blended.640px


***Readme:***
*   **Option 1:** is desirable if you need to test many subset combinations in the same session (avoid downloading data twice from the cloud).
*   **Option 2:** is desired if you are goint to work just with one dataset  (avoid downloading unnecessary data from the cloud).
*   **Option 3:** is best if you're just going to test one subset combination  (avoid downloading any data from the cloud).



In [ ]:
# Option 2 (download just the needed dataset)
cloud_path = f"/content/drive/MyDrive/YOLO/{model_name}/"
local_path = f"/content/YOLO/"
!mkdir $local_path
!cp -r $cloud_path $local_path

In [ ]:
src_folder = f"/content/YOLO/{model_name}"
data = f"{src_folder}/data.yaml"
data

'/content/YOLO/3.5m.v4i.yolov8_blended.640px/data.yaml'

## Download model

In [66]:
from ultralytics import YOLO

In [67]:
# Load pretrain YOLO v8 model
model = YOLO("yolov8m.pt")

In [ ]:
# Random intialization of YOLO v8 model
model_rnd = YOLO("yolov8m.yaml")

# Finetuning

### Optimization

In [69]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [79]:
# Garbage collection
import gc
torch.cuda.empty_cache()
gc.collect()

0

In [71]:
# Reduce VRAM usage by reducing fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

### Info

In [20]:
!nvidia-smi

Sat May  3 15:03:07 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [21]:
!yolo version

8.3.124


-----
## Experiment 52
### *YOLOv8 Mid | False color images*
False color images are created by applying:
1. Excess Green to a grayscale image.
1. A 2-component PCA (to reduce dimensionality).
1. Combining these images as RGB channels.

Including NO FREEZE

### Train

In [47]:
# Set's maximum training time (in hours)
time: float = 2 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [55]:
# Train model
history = model.train(
    data=data,
    val = True,
    epochs=500,
    imgsz=640,
    batch=32,
    #freeze=10,
    patience=200,
    time = time,
)

Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8m.pt, data=/content/YOLO/3.5m.v4i.yolov8_blended.640px/data.yaml, epochs=500, time=2, patience=200, batch=32, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, sh

train: Scanning /content/YOLO/3.5m.v4i.yolov8_blended.640px/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 707.9±515.5 MB/s, size: 159.0 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8_blended.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]


...

    246/597      11.9G       1.04     0.5723     0.9534        429        640: 100%|██████████| 9/9 [00:08<00:00,  1.01it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:01<00:00,  1.20it/s]

                   all        108       3472      0.495      0.469      0.418      0.133
EarlyStopping: Training stopped early as no improvement observed in last 200 epochs. Best results observed at epoch 46, best model saved as best.pt.
To update EarlyStopping(patience=200) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



246 epochs completed in 0.825 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 52.1MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 52.1MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:03<00:00,  1.88s/it]


                   all        108       3472      0.515      0.459      0.442      0.147
Speed: 0.2ms preprocess, 11.8ms inference, 0.0ms loss, 3.7ms postprocess per image
Results saved to runs/detect/train2


In [56]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7db702386450>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [57]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8_blended.640px/data.yaml',
          epochs=500,
          time=2,
          patience=200,
          batch=32,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train2',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=False,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.0,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
          iou=

In [58]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train2


### Save results

In [59]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save1/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save1/


-----
## Experiment 53
### *YOLOv8 Mid | False color images with CLAHE*
False color images are created by applying:
1. Excess Green to a grayscale image.
1. A 2-component PCA (to reduce dimensionality).
1. Combining these images as RGB channels.
1. CLAHE applied to tiles

Including NO FREEZE

### Train

In [74]:
# Set's maximum training time (in hours)
time: float = 2 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [80]:
# Train model
history = model.train(
    data=data,
    val = True,
    epochs=500,
    imgsz=640,
    batch=32,
    #freeze=10,
    patience=200,
    time = time,
)

New https://pypi.org/project/ultralytics/8.3.125 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8m.pt, data=/content/YOLO/3.5m.v4i.yolov8_blended.640px_clahe/data.yaml, epochs=500, time=2, patience=200, batch=32, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train3, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=Fals

train: Scanning /content/YOLO/3.5m.v4i.yolov8_blended.640px_clahe/train/labels.cache... 270 images, 0 backgrounds, 0 corrupt: 100%|██████████| 270/270 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 729.6±523.9 MB/s, size: 191.7 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8_blended.640px_clahe/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]


...

    230/484        13G      1.056     0.5744     0.9498        438        640: 100%|██████████| 9/9 [00:09<00:00,  1.04s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:01<00:00,  1.18it/s]

                   all        108       3472      0.503      0.418       0.38      0.123
EarlyStopping: Training stopped early as no improvement observed in last 200 epochs. Best results observed at epoch 30, best model saved as best.pt.
To update EarlyStopping(patience=200) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



230 epochs completed in 0.953 hours.
Optimizer stripped from runs/detect/train3/weights/last.pt, 52.1MB
Optimizer stripped from runs/detect/train3/weights/best.pt, 52.1MB

Validating runs/detect/train3/weights/best.pt...
Ultralytics 8.3.124 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:04<00:00,  2.22s/it]


                   all        108       3472      0.484      0.465      0.442      0.146
Speed: 0.2ms preprocess, 11.2ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to runs/detect/train3


In [81]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7db701acfd90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [82]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.pt',
          data='/content/YOLO/3.5m.v4i.yolov8_blended.640px_clahe/data.yaml',
          epochs=500,
          time=2,
          patience=200,
          batch=32,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device=None,
          workers=8,
          project=None,
          name='train3',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=False,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.0,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
        

In [83]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train3


### Save results

In [84]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save2/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save2/


-----
## Experiment 54
### *YOLOv8 Mid | No pre-training (random weights)*
Initialize a model with randomized weights.

### Train

In [ ]:
# Set's maximum training time (in hours)
time: float = 2 # Depending on remaining Colab cuota
# This overrides the epochs argument, allowing training to automaticallystop after the specified duration.
# Useful for time-constrained training scenarios. (aka Colab cuotas)

In [ ]:
# Train model
history = model_rnd.train(
    data=data,
    val = True,
    epochs=1000,
    imgsz=640,
    batch=-1,
    #freeze=10,
    patience=300,
    time = time,
)

Ultralytics 8.3.127 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO/3.5m.v4i.yolov8_blended.640px/data.yaml, degrees=0.0, deterministic=True, device=cuda:0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1000, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=300, perspective=0.0, pl

100%|██████████| 755k/755k [00:00<00:00, 23.0MB/s]

Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics

 18                  -1  2   1846272  ultralytics.nn.modules.block.C2f             [576, 384, 2]                 
 19                  -1  1   1327872  ultralytics.nn.modules.conv.Conv             [384, 384, 3, 2]              
 20             [-1, 9]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 21                  -1  2   4207104  ultralytics.nn.modules.block.C2f             [960, 576, 2]                 
 22        [15, 18, 21]  1   3776275  ultralytics.nn.modules.head.Detect           [1, [192, 384, 576]]          
YOLOv8m summary: 169 layers, 25,856,899 parameters, 25,856,883 gradients, 79.1 GFLOPs

Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...


100%|██████████| 5.35M/5.35M [00:00<00:00, 94.2MB/s]


...

    453/617      7.37G      1.099     0.5906      1.033        136        640: 100%|██████████| 15/15 [00:09<00:00,  1.66it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:01<00:00,  1.81it/s]

                   all        108       3472      0.476      0.416      0.375      0.117
EarlyStopping: Training stopped early as no improvement observed in last 300 epochs. Best results observed at epoch 153, best model saved as best.pt.
To update EarlyStopping(patience=300) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



453 epochs completed in 1.470 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 52.1MB
Optimizer stripped from runs/detect/train/weights/best.pt, 52.1MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.127 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8m summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:06<00:00,  2.33s/it]


                   all        108       3472      0.533      0.459      0.447      0.146
Speed: 0.3ms preprocess, 11.7ms inference, 0.0ms loss, 6.2ms postprocess per image
Results saved to runs/detect/train


In [ ]:
history

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7cb8a25de0d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Show the hyperparameters set
model.trainer.validator.args

namespace(task='detect',
          mode='train',
          model='yolov8m.yaml',
          data='/content/YOLO/3.5m.v4i.yolov8_blended.640px/data.yaml',
          epochs=1000,
          time=2,
          patience=300,
          batch=19,
          imgsz=640,
          save=True,
          save_period=-1,
          cache=False,
          device='cuda:0',
          workers=8,
          project=None,
          name='train',
          exist_ok=False,
          pretrained=True,
          optimizer='auto',
          verbose=True,
          seed=0,
          deterministic=True,
          single_cls=False,
          rect=False,
          cos_lr=False,
          close_mosaic=10,
          resume=False,
          amp=True,
          fraction=1.0,
          profile=False,
          freeze=None,
          multi_scale=False,
          overlap_mask=True,
          mask_ratio=4,
          dropout=0.0,
          val=True,
          split='val',
          save_json=False,
          conf=0.001,
        

In [ ]:
print(f"Saved into: {history.save_dir}")

Saved into: runs/detect/train


### Validation

In [ ]:
# Load currently trained YOLO model
model = YOLO(f"/content/{history.save_dir}/weights/best.pt")

In [ ]:
# Validate the model
results = model.val(data=data,
          batch=64,
          conf=0.224, # best values found with optuna
          iou=0.57, # best values found with optuna
          verbose=True,
          save_json=True)

Ultralytics 8.3.127 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8m summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2801.6±1107.7 MB/s, size: 184.0 KB)


val: Scanning /content/YOLO/3.5m.v4i.yolov8_blended.640px/valid/labels.cache... 108 images, 0 backgrounds, 0 corrupt: 100%|██████████| 108/108 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:07<00:00,  3.56s/it]


                   all        108       3472      0.528      0.477      0.486      0.179
Speed: 7.0ms preprocess, 23.6ms inference, 0.0ms loss, 3.2ms postprocess per image
Saving runs/detect/val/predictions.json...
Results saved to runs/detect/val


In [ ]:
print(f"Saved into: {results.save_dir}")

Saved into: runs/detect/val


In [ ]:
percentages = gimme_metrics(results)

Total objects detected: 4738.0
Confusion matrix:
['39.53%', '26.72%']
['33.75%', '0.00%']


In [ ]:
save_json(results)

✅ JSON file stored in: runs/detect/val


### Save results

In [ ]:
# Store model weights and metrics
save_on_cloud(source='/content/runs/', destination='/content/drive/MyDrive/save1/')

✅ Folder copied successfully:
   /content/runs/ 
  --> /content/drive/MyDrive/save1/


#### Classification metrics validation

In [ ]:
percentages

[['39.53%', '26.72%'], ['33.75%', '0.00%']]

In [ ]:
# Setting values from CM graph
TP = 4738 *0.3953
FP = 4738 *0.2672
FN = 4738 *0.3375

In [ ]:
# Confusion matrix
show_metrics(TP, FP, FN)

Total objects detected: 4738.0

Confusion matrix:
[ 39.53% , 26.72% ]
[ 33.75% , 0.00% ]

Metrics:
- Accuracy: 0.395
- Precision: 0.597
- Recall: 0.539
- F1 Score: 0.567
- F½ Score: 0.584
- G-mean: 0.567
